# Praktikum 4 - Perbandingan Annoy, FAISS, dan HNSW

**Nama:** Radit Firansah  
**NIM:** 244107020196

Membandingkan build time dan query time ketiga library pada dataset 1 juta data 5D.

In [1]:
import numpy as np
import time
from annoy import AnnoyIndex
import faiss
import hnswlib

n_data = 1_000_000
dim = 5
X = np.random.random((n_data, dim)).astype(np.float32)

query = np.random.random((1, dim)).astype(np.float32)
k = 10

### Annoy

In [2]:
ann_index = AnnoyIndex(dim, 'euclidean')

start = time.time()
for i in range(n_data):
    ann_index.add_item(i, X[i])
ann_index.build(10)
build_time = time.time() - start

start = time.time()
neighbors = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", neighbors[0][:5], "...")

Build time: 2.160325527191162 detik
Query time: 0.00010919570922851562 detik
Neighbors: [1321, 639661, 945283, 306501, 133290] ...


### FAISS (IndexFlatL2)

In [3]:
faiss_index = faiss.IndexFlatL2(dim)

start = time.time()
faiss_index.add(X)
build_time = time.time() - start

start = time.time()
distances, indices = faiss_index.search(query, k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", indices[0][:5], "...")

Build time: 0.008001089096069336 detik
Query time: 0.005537986755371094 detik
Neighbors: [  1321 639661 945283 306501 133290] ...


### HNSW (hnswlib)

In [4]:
hnsw_index = hnswlib.Index(space='l2', dim=dim)

start = time.time()
hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
hnsw_index.add_items(X)
build_time = time.time() - start

hnsw_index.set_ef(50)

start = time.time()
labels, distances = hnsw_index.knn_query(query, k=k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", labels[0][:5], "...")

Build time: 13.514862775802612 detik
Query time: 0.00010848045349121094 detik
Neighbors: [  1321 639661 945283 306501 133290] ...


## Tabel Hasil Percobaan (Metric Berbeda)

In [5]:
import numpy as np
import time
import pandas as pd
from annoy import AnnoyIndex
import faiss
import hnswlib

n_data = 1_000_000
dim = 5
np.random.seed(42)
X = np.random.random((n_data, dim)).astype(np.float32)
query = np.random.random((1, dim)).astype(np.float32)
k = 10

rows = []

for metric in ['euclidean', 'angular']:
    ann_index = AnnoyIndex(dim, metric)
    start = time.time()
    for i in range(n_data):
        ann_index.add_item(i, X[i])
    ann_index.build(10)
    build = time.time() - start
    start = time.time()
    neighbors = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
    qtime = time.time() - start
    rows.append({'Library': 'Annoy', 'Metric': metric,
                 'Build (s)': round(build, 3), 'Query (s)': round(qtime, 6),
                 'Neighbors': neighbors[0][:5]})

for name, m in [('L2 (Euclidean)', faiss.METRIC_L2), ('Inner Product', faiss.METRIC_INNER_PRODUCT)]:
    faiss_index = faiss.IndexFlat(dim, m)
    start = time.time()
    faiss_index.add(X)
    build = time.time() - start
    start = time.time()
    distances, indices = faiss_index.search(query, k)
    qtime = time.time() - start
    rows.append({'Library': 'FAISS Flat', 'Metric': name,
                 'Build (s)': round(build, 3), 'Query (s)': round(qtime, 6),
                 'Neighbors': list(indices[0][:5])})

for space in ['l2', 'ip']:
    hnsw_index = hnswlib.Index(space=space, dim=dim)
    start = time.time()
    hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
    hnsw_index.add_items(X)
    build = time.time() - start
    hnsw_index.set_ef(50)
    start = time.time()
    labels, distances = hnsw_index.knn_query(query, k=k)
    qtime = time.time() - start
    rows.append({'Library': 'HNSW', 'Metric': space,
                 'Build (s)': round(build, 3), 'Query (s)': round(qtime, 6),
                 'Neighbors': list(labels[0][:5])})

df_hasil = pd.DataFrame(rows)
df_hasil

,Library,Metric,Build (s),Query (s),Neighbors
0,Annoy,euclidean,2.148,0.000073,"[901095, 495168, 561017, 163986, 185734]"
1,Annoy,angular,2.665,0.000066,"[653774, 135588, 561017, 901095, 211473]"
2,FAISS Flat,L2 (Euclidean),0.008,0.009054,"[901095, 495168, 561017, 163986, 185734]"
3,FAISS Flat,Inner Product,0.002,0.004495,"[958577, 238618, 509273, 52984, 409344]"
4,HNSW,l2,14.467,0.000061,"[901095, 495168, 561017, 163986, 185734]"
5,HNSW,ip,14.323,0.000062,"[958577, 238618, 509273, 52984, 409344]"
